Plotting AUC against Trials and saving as PDF

In [0]:
from pyspark.sql import Row

data = [
    Row(Trial=1,  AUC=0.6525), #Logistic Regression 1, 5.mars
    Row(Trial=2,  AUC=0.6251), #Random Forest 1, 5.mars
    Row(Trial=3,  AUC=0.5383), #Gradient Boosted Trees 1, 5.mars
    Row(Trial=4,  AUC=0.9115), #Logistic Regression 2, 8.mars
    Row(Trial=5,  AUC=0.9139), #Logistic Regression 3, 8.mars
    Row(Trial=6,  AUC=0.9147), #Gradient Boosted Trees 2, 8.mars
    Row(Trial=7,  AUC=0.9114), #Logistic Regression 4, 12.mars
    Row(Trial=8,  AUC=0.9005), #Logistic Regression 5, 12.mars
    Row(Trial=9,  AUC=0.9237), #Logistic Regression 6, 12.mars
    Row(Trial=10, AUC=0.9350), #Logistic Regression 7, 12.mars
    Row(Trial=11, AUC=0.9411), #Random Forest 2, 12.mars
    Row(Trial=12, AUC=0.9156), #Gradient Boosted Trees 3, 12.mars
    Row(Trial=13, AUC=0.9240), #Random Forest 3, 15.mars
    Row(Trial=14, AUC=0.9360), #Gradient Boosted Trees 5, 17.mars
    Row(Trial=15, AUC=0.9379), #Multilayer Perceptron 1, 23.mars
    Row(Trial=16, AUC=0.9238), #Multilayer Perceptron 2, 29.mars
    Row(Trial=17, AUC=0.9136), #Multilayer Perceptron 3, 29.mars
    Row(Trial=18, AUC=0.9137), #Multilayer Perceptron 4, 29.mars
    Row(Trial=19, AUC=0.9034), #Multilayer Perceptron 5, 29.mars
    Row(Trial=20, AUC=0.8837), #Multilayer Perceptron 6, 29.mars
    Row(Trial=21, AUC=0.8847), #Multilayer Perceptron 7, 29.mars
    Row(Trial=22, AUC=0.9314), #Multilayer Perceptron 8, 29.mars
    Row(Trial=23, AUC=0.9452), #Multilayer Perceptron 9, 2.apríl
]

df_auc = spark.createDataFrame(data)
display(df_auc)

Databricks visualization. Run in Databricks to view.

In [0]:
import plotly.graph_objects as go

# Trial data with model types
trials = [
    (1,  0.6525, "LR"),  (2,  0.6251, "RF"),  (3,  0.5383, "GBT"),
    (4,  0.9115, "LR"),  (5,  0.9139, "LR"),  (6,  0.9147, "GBT"),
    (7,  0.9114, "LR"),  (8,  0.9005, "LR"),  (9,  0.9237, "LR"),
    (10, 0.9350, "LR"),  (11, 0.9411, "RF"),  (12, 0.9156, "GBT"),
    (13, 0.9240, "RF"),  (14, 0.9360, "GBT"), (15, 0.9379, "MLP"),
    (16, 0.9238, "MLP"), (17, 0.9136, "MLP"), (18, 0.9137, "MLP"),
    (19, 0.9034, "MLP"), (20, 0.8837, "MLP"), (21, 0.8847, "MLP"),
    (22, 0.9314, "MLP"), (23, 0.9452, "MLP"),
]

trial_nums = [t[0] for t in trials]
aucs = [t[1] for t in trials]
model_types = [t[2] for t in trials]

colors = {"LR": "#3498db", "RF": "#2ecc71", "GBT": "#e67e22", "MLP": "#9b59b6"}
full_names = {"LR": "Logistic Regression", "RF": "Random Forest", "GBT": "Gradient Boosted Trees", "MLP": "Multilayer Perceptron"}

fig = go.Figure()

# Connecting line
fig.add_trace(go.Scatter(
    x=trial_nums, y=aucs, mode="lines",
    line=dict(color="#cccccc", width=1.5),
    showlegend=False, hoverinfo="skip"
))

# Points colored by model type
for mt in ["LR", "RF", "GBT", "MLP"]:
    idx = [i for i, m in enumerate(model_types) if m == mt]
    fig.add_trace(go.Scatter(
        x=[trial_nums[i] for i in idx],
        y=[aucs[i] for i in idx],
        mode="markers",
        name=full_names[mt],
        marker=dict(color=colors[mt], size=10, line=dict(width=1, color="white")),
        hovertemplate="Trial %{x}<br>AUC: %{y:.4f}<extra>" + full_names[mt] + "</extra>"
    ))

# Highlight best trial
best_idx = aucs.index(max(aucs))
fig.add_annotation(
    x=trial_nums[best_idx], y=aucs[best_idx],
    text=f"Best: {max(aucs):.4f}",
    showarrow=True, arrowhead=2, arrowcolor="#333",
    ax=30, ay=-30,
    font=dict(size=12, color="#333", family="Arial Black"),
    bgcolor="#ffffcc", bordercolor="#333", borderwidth=1
)

# Phase annotations
fig.add_vrect(x0=0.5, x1=3.5, fillcolor="#f0f0f0", opacity=0.3, line_width=0,
              annotation_text="Initial\nbaseline", annotation_position="top left",
              annotation_font_size=10, annotation_font_color="#888")
fig.add_vrect(x0=3.5, x1=14.5, fillcolor="#e8f4fd", opacity=0.2, line_width=0,
              annotation_text="Feature engineering\n& tuning", annotation_position="top left",
              annotation_font_size=10, annotation_font_color="#888")
fig.add_vrect(x0=14.5, x1=23.5, fillcolor="#f3e8fd", opacity=0.2, line_width=0,
              annotation_text="MLP\nexploration", annotation_position="top left",
              annotation_font_size=10, annotation_font_color="#888")

fig.update_layout(
    title=dict(text="Model Development: AUC Across Trials", font=dict(size=16)),
    xaxis=dict(
        title=dict(text="Trial", standoff=10),
        showgrid=True, gridcolor="#eeeeee", dtick=1, range=[0, 24]
    ),
    yaxis=dict(
        title="AUC",
        showgrid=True, gridcolor="#eeeeee", range=[0.5, 1.0]
    ),
    plot_bgcolor="white",
    paper_bgcolor="white",
    legend=dict(orientation="h", yanchor="bottom", y=-0.25, xanchor="center", x=0.5),
    height=450
)
fig.show()

print(f"Best AUC: {max(aucs):.4f} (Trial {trial_nums[best_idx]}, {full_names[model_types[best_idx]]})")
print(f"AUC improvement from baseline: {max(aucs) - aucs[0]:.4f} (+{((max(aucs)/aucs[0])-1)*100:.1f}%)")

In [0]:
import subprocess, sys, importlib, base64

# Install kaleido to temp dir
target_dir = "/tmp/kaleido_pkg"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--target", target_dir, "kaleido==0.2.1"])
if target_dir not in sys.path:
    sys.path.insert(0, target_dir)

# Use kaleido directly — disable MathJax by bypassing the default setter
from kaleido.scopes.plotly import PlotlyScope
scope = PlotlyScope()
scope._mathjax = None  # Forces --mathjax flag to NOT be passed to chromium

# Export figure exactly as defined in cell 5
pdf_bytes = scope.transform(
    fig.to_dict(), format="pdf", width=1000, height=450
)
print(f"PDF generated: {len(pdf_bytes)} bytes")

# Download link
b64 = base64.b64encode(pdf_bytes).decode()
displayHTML(f'<a href="data:application/pdf;base64,{b64}" download="AUC_Across_Trials.pdf" style="font-size:16px; padding:12px 24px; background:#4CAF50; color:white; text-decoration:none; border-radius:5px;">\U0001f4e5 Download PDF for LaTeX</a>')

## Global SHAP Value calculations:

In [0]:
import shap
import numpy as np
from pyspark.sql import Row
from pyspark.ml.linalg import Vectors
from pyspark.ml.functions import vector_to_array

# Feature names (must match training feature order: sensor + lags + deviation + deviceId)
sensor_cols = [
    "payload_xrayController_filamentCurrent",
    "payload_xrayController_onTime",
    "payload_xrayController_temperature",
    "payload_xrayController_tubeCurrent",
    "payload_xrayController_tubeVoltage",
    "delta_seconds",
]
n_lags = 20
lag_cols = [f"{col}{L}" for L in range(1, n_lags + 1) for col in sensor_cols]
dev_cols = [
    "payload_xrayController_filamentCurrent_avg_daily",
    "payload_xrayController_filamentCurrent_dev_daily",
    "payload_xrayController_temperature_avg_daily",
    "payload_xrayController_temperature_dev_daily",
    "payload_xrayController_tubeCurrent_avg_daily",
    "payload_xrayController_tubeCurrent_dev_daily",
]
feature_names = sensor_cols + lag_cols + dev_cols + ["deviceId_index"]
print(f"Total features: {len(feature_names)}")

# --- Sample training data to numpy ---
train_sample = (
    train.select(vector_to_array("features").alias("features_array"))
    .orderBy(F.rand(seed=42))
    .limit(1000)
    .toPandas()
)
X_all = np.array(train_sample["features_array"].tolist())

# Background summary (50 kmeans clusters from 500 samples)
background = shap.kmeans(X_all[:500], 50)
X_explain = X_all[500:1000]  # 500 samples to explain

print(f"Background: 50 kmeans clusters from 500 samples")
print(f"Explaining: {X_explain.shape[0]} samples")
print(f"nsamples: 500 perturbations per sample")

# --- Prediction wrapper: numpy → class 1 (fault) probability ---
def predict_proba(X):
    rows = [Row(features=Vectors.dense(x.tolist())) for x in X]
    df_pred = spark.createDataFrame(rows)
    preds = (
        MLP_v4.transform(df_pred)
        .select(vector_to_array("probability").alias("prob"))
        .toPandas()
    )
    return np.array(preds["prob"].tolist())  # returns [P(class0), P(class1)] per row

# --- Compute SHAP values ---
explainer = shap.KernelExplainer(predict_proba, background)
shap_values = explainer.shap_values(X_explain, nsamples=500, silent=True)

# Handle different SHAP output formats (list vs 3D array)
if isinstance(shap_values, list):
    shap_vals_fault = shap_values[1]
elif isinstance(shap_values, np.ndarray) and shap_values.ndim == 3:
    shap_vals_fault = shap_values[:, :, 1]
else:
    shap_vals_fault = shap_values

print(f"\nSHAP values computed: {shap_vals_fault.shape}")
print(f"\nTop 15 most important features (mean |SHAP|):")
mean_abs_shap = np.abs(shap_vals_fault).mean(axis=0)
top_idx = np.argsort(mean_abs_shap)[::-1][:15]
for i, idx in enumerate(top_idx, 1):
    print(f"  {i}. {feature_names[int(idx)]}: {mean_abs_shap[int(idx)]:.4f}")

# --- Global SHAP summary plot ---
shap.summary_plot(shap_vals_fault, X_explain, feature_names=feature_names, show=False)
import matplotlib.pyplot as plt
plt.tight_layout()
plt.show()
print("SHAP summary plot displayed above")

## MLP Fault Prediction Model — Results & Evaluation
This section presents the evaluation of the **Multilayer Perceptron (MLP)** classifier trained to predict X-ray controller faults using sensor telemetry data. The model uses a **failure horizon** label (fault within the next N hours) rather than the instantaneous fault state, enabling **early warning** capability.

In [0]:
import pyspark.sql.functions as F
import pandas
from datetime import datetime, timedelta
from pyspark.sql.window import Window
import os
from marel.services import ServiceProvider
from marel.databricks import DatabricksProvider
from pyspark.ml.feature import StandardScaler, VectorAssembler
from pyspark.ml.functions import vector_to_array

from pyspark.ml.classification import MultilayerPerceptronClassifier
import mlflow

provider = DatabricksProvider()
table_service = provider.table_service
provider = DatabricksProvider()

os.environ["MLFLOW_DFS_TMP"] = "/Volumes/teams/sensorx/data-dump/mlflow_tmp"

MLP_v4 = mlflow.spark.load_model("models:/teams.sensorx.mlp_fault_prediction/4")

train = spark.read.table("teams.sensorx.train_data")
test = spark.read.table("teams.sensorx.test_data")

In [0]:

df_csv = spark.read.format("csv") \
    .option("header", True) \
    .option("inferSchema", True) \
    .option("multiLine", True) \
    .option("escape", "\"") \
    .option("quote", "\"") \
    .option("delimiter", ",") \
    .load("/Volumes/teams/sensorx/data-dump/deviceID_Serial_GenSerial_machineCode.csv")

# Device IDs that have at least one 800W serial number
devices_800w = (
    df_csv.filter(F.col("gentype") == "800W")
    .select("properties_deviceId")
    .distinct()
)

In [0]:
def df_read_data(days):

    # Read telemetry data
    df_telemetry = table_service.load_table_from_device_checkpoint(
        table_name = "prod_medallion.silver_machine.sensorx_telemetry",
        machine_name="sensorx",
        max_historical_days=days,
        old_format=True
    ).select("properties_deviceID",
            "properties_payloadTimeStamp",
            "payload_xrayController_filamentCurrent",
            "payload_xrayController_onTime",
            "payload_xrayController_temperature",
            "payload_xrayController_tubeCurrent",
            "payload_xrayController_tubeVoltage"
            )

    # Filter to 800w devices
    df_telemetry_800w = df_telemetry.join(
        F.broadcast(devices_800w), df_telemetry["properties_deviceID"] == devices_800w["properties_deviceId"], "inner"
    ).drop(devices_800w["properties_deviceId"])

    # Reading fault tables
    df_fault = (
        table_service.load_table_from_device_checkpoint(
            table_name = "prod_medallion.silver_machine.pluto_xraycontroller_fault_property",
            machine_name="sensorx",
            max_historical_days=days,
            old_format=True
        )
        .filter(F.col("payload_fault_faultName") == "faultRegulation")
        .select("properties_payloadTimeStamp",
                "properties_deviceID",
                "payload_fault_state")
    )

    # --- Union + forward-fill ---
    telem_only = [c for c in df_telemetry_800w.columns
                if c not in ("properties_payloadTimeStamp", "properties_deviceID")]

    telem_part = df_telemetry_800w.select(
        "properties_payloadTimeStamp", "properties_deviceID",
        *telem_only,
        F.lit(None).cast("boolean").alias("payload_fault_state"),
        F.lit(True).alias("_is_telem"),
    )

    fault_part = df_fault.select(
        "properties_payloadTimeStamp", "properties_deviceID",
        *[F.lit(None).cast(dict(df_telemetry_800w.dtypes)[c]).alias(c)
        for c in telem_only],
        "payload_fault_state",
        F.lit(False).alias("_is_telem"),
    )

    w = Window.partitionBy("properties_deviceID").orderBy("properties_payloadTimeStamp")

    df = (
        telem_part.unionByName(fault_part)
        .withColumn("payload_fault_state",
                    F.last("payload_fault_state", ignorenulls=True).over(w))
        .filter(F.col("_is_telem"))
        .drop("_is_telem")
    )

    df = df.withColumn(
        "payload_fault_state", F.coalesce(F.col("payload_fault_state"), F.lit(False))
    )

    # Adding delta seconds feature
    df = (
        df
        .withColumn("prev_ts", F.lag("properties_payloadTimeStamp").over(w))
        .withColumn(
            "delta_seconds",
            F.when(
                F.col("prev_ts").isNull() |
                (F.col("properties_payloadTimeStamp").cast("long") - F.col("prev_ts").cast("long") < 0),
                None
            ).otherwise(
                F.col("properties_payloadTimeStamp").cast("long") - F.col("prev_ts").cast("long")
            ).cast("double")
        )
        .drop("prev_ts")
    )

    # Feature engineering
    sensor_cols = [c for c in df.columns
                   if c not in {"properties_deviceID", "properties_payloadTimeStamp", "payload_fault_state"}]

    df = df.na.drop(subset=sensor_cols)

    # Encode deviceID as integer index
    device_ids = df.select("properties_deviceID").distinct().orderBy("properties_deviceID")
    device_ids = device_ids.withColumn(
        "deviceId_index",
        (F.dense_rank().over(Window.orderBy("properties_deviceID")) - 1).cast("double")
    )
    df = df.join(device_ids, on="properties_deviceID", how="inner")

    # Lag features
    n_lags = 20
    w_lag = Window.partitionBy("properties_deviceID").orderBy("properties_payloadTimeStamp")

    lag_exprs = [F.lag(col_name, L).over(w_lag).alias(f"{col_name}{L}")
                 for L in range(1, n_lags + 1) for col_name in sensor_cols]
    df = df.select("*", *lag_exprs)

    lag_cols = [f"{col}{L}" for L in range(1, n_lags + 1) for col in sensor_cols]

    df = df.na.drop(subset=lag_cols)

    # --- Deviation features: 24-hour time-based rolling average ---
    w_daily = (
        Window.partitionBy("properties_deviceID")
        .orderBy(F.col("properties_payloadTimeStamp").cast("long"))
        .rangeBetween(-86400, 0)
    )

    dev_sensors = [
        "payload_xrayController_filamentCurrent",
        "payload_xrayController_temperature",
        "payload_xrayController_tubeCurrent",
    ]
    dev_cols = []
    dev_exprs = []

    for col_name in dev_sensors:
        avg_col = f"{col_name}_avg_daily"
        dev_col = f"{col_name}_dev_daily"
        dev_cols.extend([avg_col, dev_col])
        dev_exprs.append(F.avg(col_name).over(w_daily).alias(avg_col))
        dev_exprs.append((F.col(col_name) - F.avg(col_name).over(w_daily)).alias(dev_col))

    df = df.select("*", *dev_exprs)

    # Assemble features (133 total: 6 sensor + 120 lags + 6 deviation + 1 deviceId_index)
    feature_input_cols = sensor_cols + lag_cols + dev_cols + ["deviceId_index"]
    assembler = VectorAssembler(inputCols=feature_input_cols, outputCol="features")
    df_features = assembler.transform(df)

    df_features = df_features.select("properties_payloadTimeStamp", "properties_deviceID", "features")

    # --- Normalize using training data statistics (must match training scaler) ---
    train_data = spark.read.table("teams.sensorx.train_data").select("features")

    scaler = StandardScaler(inputCol="features", outputCol="features_scaled", withMean=True, withStd=True)
    scaler_model = scaler.fit(train_data)  # fit on TRAINING data, not inference data

    df_features_norm = scaler_model.transform(df_features).drop("features").withColumnRenamed("features_scaled", "features")

    return df_features_norm

In [0]:
days = 90
data = df_read_data(days)
display(data.limit(10))

In [0]:
predictions = MLP_v4.transform(data)

# Simplify output: extract fault probability and drop vector columns
results = predictions.select(
    "properties_payloadTimeStamp",
    "properties_deviceID",
    F.col("prediction").cast("int").alias("predicted_label"),
    F.round(vector_to_array("probability")[1], 4).alias("fault_probability"),
).cache()

results.count()  # materialize cache so downstream cells are instant

In [0]:
%pip install shap -q

In [0]:
# === Maintenance Decision Table ===
# Latest 24h predictions → which machines are at risk of failing within the next 15 days (our horizon)

max_ts = results.agg(F.max("properties_payloadTimeStamp")).collect()[0][0]

w_rank = Window.partitionBy("properties_deviceID").orderBy(F.col("fault_probability").desc())

maintenance_table = (
    results
    .filter(F.col("properties_payloadTimeStamp") >= F.lit(max_ts) - F.expr("INTERVAL 24 HOURS"))
    .withColumn("rn", F.row_number().over(w_rank))
    .filter(F.col("rn") == 1)
    .drop("rn", "properties_payloadTimeStamp", "predicted_label")
    .withColumnRenamed("properties_deviceID", "machine_id")
    .withColumn("risk_level",
        F.when(F.col("fault_probability") >= 0.80, "High")
         .when(F.col("fault_probability") >= 0.50, "Medium")
         .otherwise("Low")
    )
    .withColumn("recommended_action",
        F.when(F.col("fault_probability") >= 0.80, "Inspect immediately")
         .when(F.col("fault_probability") >= 0.50, "Schedule maintenance")
         .otherwise("Continue monitoring")
    )
    .select("machine_id", "fault_probability", "risk_level", "recommended_action")
    .orderBy(F.col("fault_probability").desc())
)

display(maintenance_table)

In [0]:
# === Top 10 Highest-Priority Machines ===
top10 = result.head(10)

print("=" * 100)
print("  TOP 10 HIGHEST-PRIORITY MACHINES — MAINTENANCE ACTION REQUIRED")
print("=" * 100)

for i, (_, row) in enumerate(top10.iterrows(), 1):
    print(f"\n  {i}. Machine:  {row['machine_id']}")
    print(f"     Risk:     {row['risk_level']} ({row['failure_probability']:.2%})")
    print(f"     Action:   {row['recommended_action']}")
    print(f"     Drivers:  {row['main_reason']}")

print("\n" + "=" * 100)
print("  Model: MLP Fault Prediction v3  |  Explanations: SHAP (top 3 features)")
print("=" * 100)

In [0]:
# === Visual Summary of Maintenance Decisions ===
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --- Chart 1: Failure Probability per Machine (color-coded by risk) ---
df_plot = result.copy()
df_plot["machine_short"] = df_plot["machine_id"].str[:8] + "..."
df_plot = df_plot.sort_values("failure_probability", ascending=True)  # bottom-to-top

color_map = {"High": "#D94040", "Medium": "#E8A838", "Low": "#4CAF82"}
colors = [color_map[r] for r in df_plot["risk_level"]]

fig1 = go.Figure()
fig1.add_trace(go.Bar(
    x=df_plot["failure_probability"],
    y=df_plot["machine_short"],
    orientation="h",
    marker_color=colors,
    text=[f"{p:.1%}" for p in df_plot["failure_probability"]],
    textposition="outside",
    hovertext=[
        f"Machine: {mid}<br>Risk: {rl}<br>Probability: {fp:.4%}<br>Action: {ra}"
        for mid, rl, fp, ra in zip(
            df_plot["machine_id"], df_plot["risk_level"],
            df_plot["failure_probability"], df_plot["recommended_action"]
        )
    ],
    hoverinfo="text",
))

# Add risk level threshold lines
fig1.add_vline(x=0.80, line_dash="dash", line_color="#D94040", line_width=1,
               annotation_text="High (80%)", annotation_position="top")
fig1.add_vline(x=0.50, line_dash="dash", line_color="#E8A838", line_width=1,
               annotation_text="Medium (50%)", annotation_position="top")

fig1.update_layout(
    title="Failure Probability by Machine (15-Day Horizon)",
    xaxis_title="Failure Probability",
    yaxis_title="Machine ID",
    xaxis=dict(range=[0, 1.12], tickformat=".0%", showgrid=True, gridcolor="#EEEEEE"),
    yaxis=dict(showgrid=False),
    plot_bgcolor="white",
    paper_bgcolor="white",
    height=500,
    margin=dict(l=100),
)
fig1.show()

